# Quick question
Have you used
- dplyr
- Quarto?
- HTML?
- CSS?
- JS?
- What else?

# Pipe operator
Function composition is better written with the pipe ("then") operator `|>`

In [2]:
# With usual function composition notation
mean(head(penguins$body_mass, 3))

[1] 3600

In [10]:
# With pipe notation
penguins$body_mass |> head(3) |> mean()

[1] 3600

`A |> f |> g |> ...` plugs `A` into the first argument of `f`, then takes that output and plugs it into the first argument of `g`, so on.

To pipe into a given named argument, use `_` in the desired slot:

In [13]:
3|> head(c(10, 20, 30, 40, 50, 60), n = _)

[1] 10 20 30

# Tidy data and `tidyverse`

Data is **tidy** if
- each variable is in its own column,
- each observation is in its own row,
- each value is in its own cell.

<img src="https://statsandr.com/blog/introduction-to-data-manipulation-in-r-with-dplyr/images/tidy-data.png" width="850">


Left table below is in tidy format since each variable is a cloumn.

<img src="https://yardbook.jhelvy.com/images/data-shapes.png" width="600">

Overall `tidyverse` library is a collection of packages that follow a set of conventions like:
- tables are tidy, especially in `tibble` format.
- names use `_` and not the `.` convention from base R.
- functions use action verb naming.

## `tibble`

In a data frame, the rows can be named or have an ID.

We will use instead use a `tibble`, where rows cannot be directly named: instead the name or ID must be put as a variable.

`tibble` is the `tidyverse`'s version of a data frame without some edge cases.

In [14]:
penguins[1,] |> head()
penguins[,1] |> head()

,species,island,bill_len,bill_dep,flipper_len,body_mass,sex,year
,<fct>,<fct>,<dbl>,<dbl>,<int>,<int>,<fct>,<int>
1,Adelie,Torgersen,39.1,18.7,181,3750,male,2007


[1] Adelie Adelie Adelie Adelie Adelie Adelie
Levels: Adelie Chinstrap Gentoo

In [16]:
library(tidyverse)
df <- as_tibble(penguins)
df[1,] |> head()
df[,1] |> head()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


species,island,bill_len,bill_dep,flipper_len,body_mass,sex,year
<fct>,<fct>,<dbl>,<dbl>,<int>,<int>,<fct>,<int>
Adelie,Torgersen,39.1,18.7,181,3750,male,2007


species
<fct>
Adelie
Adelie
Adelie
Adelie
Adelie
Adelie


# `dplyr` manipulates data

In [ ]:
# Load in `dplyr` library or `tidyverse` which includes `dplyr` and more.
# library(tidyverse)
library(dplyr)
library(tidyr)

In [17]:
# From https://www.kaggle.com/datasets/dhruvildave/billboard-the-hot-100-songs
read.csv("https://manctsui.github.io/datasets/charts.csv") |> as_tibble() -> df
df |> head()

date,rank,song,artist,last.week,peak.rank,weeks.on.board
<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>
2021-11-06,1,Easy On Me,Adele,1,1,3
2021-11-06,2,Stay,The Kid LAROI & Justin Bieber,2,1,16
2021-11-06,3,Industry Baby,Lil Nas X & Jack Harlow,3,1,14
2021-11-06,4,Fancy Like,Walker Hayes,4,3,19
2021-11-06,5,Bad Habits,Ed Sheeran,5,2,18
2021-11-06,6,Way 2 Sexy,Drake Featuring Future & Young Thug,6,1,8


## Select columns and filter rows

Variables are named, so selecting columns is more string matching than complex logical conditions:

- Do `|> select(column_name_1,column_name_2,...)`

Observations come in many data types, so filtering rows involves complex logical conditions:

- Do `|> filter(row_condition_1, row_condition_2,...)` to AND row conditions together.

**Example 1A.** Report the song name, last week rank, and weeks on board for those songs with with missing `last.week` entries.

In [22]:
df |>
  select(song, last.week, weeks.on.board) |>
  filter(is.na(last.week))

song,last.week,weeks.on.board
<chr>,<int>,<int>
Moth To A Flame,NA,1
Let's Go Brandon,NA,1
Not In The Mood,NA,1
Switches & Dracs,NA,1
Poke It Out,NA,1
Scorpio,NA,1
Big Energy,NA,1
Half Of My Hometown,NA,1
Money,NA,1


**Example 1B.** In above example, filter further to see if the songs with missing last week rank could have been on the board for more than 1 week.

In [25]:
df |>
  select(song, last.week, weeks.on.board) |>
  filter(is.na(last.week), weeks.on.board > 1)

song,last.week,weeks.on.board
<chr>,<int>,<int>
Ain't Shit,NA,14
Life Goes On,NA,2
Come Through,NA,16
Nevada,NA,4
Maybach,NA,3
Woo Baby,NA,2
Ain't Shit,NA,13
For Tonight,NA,2
Praise God,NA,3


Common `select` patterns, including use with helper functions:
- `|> select(col_A:col_Z)` select columns between `col_A` and `col_Z` inclusive.
- `|> select(contains("string"))` selects column names containing "string"
- `|> select(matches(".t"))` selects column names matching regular expression ".t"
- `|> select(starts_with("string"))` selects column names starting with "string"
- `|> select(ends_with("string"))` selects column names ending with "string"
- `|> select(where(is.numeric))` selects numeric columns
- `|> select(col_A, col_B, everything())` reorders columns so that `col_A` and `col_B` come first: `everything` fills in with the rest of the columns.

## More column operations: `mutate`
`|> mutate(column = formula)` writes over old column name or writes a new column according to formula.

**Example.** Add a column which states number of days the song is on the charts.

In [26]:
df |>
  mutate(days.on.board = 7 * weeks.on.board)

date,rank,song,artist,last.week,peak.rank,weeks.on.board,days.on.board
<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>,<dbl>
2021-11-06,1,Easy On Me,Adele,1,1,3,21
2021-11-06,2,Stay,The Kid LAROI & Justin Bieber,2,1,16,112
2021-11-06,3,Industry Baby,Lil Nas X & Jack Harlow,3,1,14,98
2021-11-06,4,Fancy Like,Walker Hayes,4,3,19,133
2021-11-06,5,Bad Habits,Ed Sheeran,5,2,18,126
2021-11-06,6,Way 2 Sexy,Drake Featuring Future & Young Thug,6,1,8,56
2021-11-06,7,Shivers,Ed Sheeran,9,7,7,49
2021-11-06,8,Good 4 U,Olivia Rodrigo,7,1,24,168
2021-11-06,9,Need To Know,Doja Cat,11,9,20,140


# More row operations
Common row manipulation patterns:
- `|> drop_na(col_A, col_B)` removes all NA values from `colA` and `col_B` from `tidyr`.
- `|> distinct(col_A, col_B)` removes all duplicate rows occurring as `(col_A, col_B)`.
- `|> arrange(col_A, desc(col_B))` sorts by `col_A` ascending and then `col_B` descending.
- `|> slice(your_vector)` returns the data frame or tibble of rows indexed by `your_vector`.

**Example.** Return songs by Olivia Rodrigo, with no repeated songs, with newest songs first.

In [38]:
df |>
  filter(artist == 'Olivia Rodrigo') |>
  arrange(desc(date), weeks.on.board) |>
  distinct(song, .keep_all = TRUE)

date,rank,song,artist,last.week,peak.rank,weeks.on.board
<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>
2021-11-06,19,Traitor,Olivia Rodrigo,22,9,23
2021-11-06,8,Good 4 U,Olivia Rodrigo,7,1,24
2021-10-30,50,Deja Vu,Olivia Rodrigo,42,3,29
2021-09-11,97,Brutal,Olivia Rodrigo,89,12,12
2021-08-28,93,Favorite Crime,Olivia Rodrigo,89,16,13
2021-08-21,97,Happier,Olivia Rodrigo,93,15,12
2021-07-31,99,"Jealousy, Jealousy",Olivia Rodrigo,87,24,9
2021-07-31,43,Drivers License,Olivia Rodrigo,42,1,28
2021-07-17,98,Enough For You,Olivia Rodrigo,NA,14,6


**Example.** Remove all NA values from `last.week` and `weeks.on.board`, arrange by least to most weeks on board, and remove repeated songs.

In [42]:
df |>
  drop_na(last.week, weeks.on.board) |>
  arrange(weeks.on.board) |>
  distinct(song, .keep_all = T)

date,rank,song,artist,last.week,peak.rank,weeks.on.board
<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>
2021-11-06,35,Better Days,NEIKED X Mae Muller X Polo G,57,35,2
2021-11-06,38,Lets Go Brandon,Loza Alexander,45,38,2
2021-11-06,45,Bubbly,Young Thug With Drake & Travis Scott,20,20,2
2021-11-06,62,Pissed Me Off,Lil Durk,39,39,2
2021-11-06,92,"Ya Superame (En Vivo Desde Culiacan, Sinaloa)",Grupo Firme,92,92,2
2021-10-30,1,Easy On Me,Adele,68,1,2
2021-10-30,19,Who Want Smoke??,"Nardo Wick Featuring G Herbo, Lil Durk & 21 Savage",17,17,2
2021-10-30,67,WFM,Realestk,75,67,2
2021-10-30,73,Lo Siento BB:/,"Tainy, Bad Bunny & Julieta Venegas",51,51,2


# Grouping and summarizing

`|> group_by(col_A, col_B)` groups data so that `col_A, col_B` are together.

`|> summarize(fn_1(col_A), fn_2(col_B))` summarizes `col_A` using `fn_1` and `col_B` using `fn_2`.

**Example.** Group by artists, then by songs.

In [43]:
df |>
  group_by(artist, song)

date,rank,song,artist,last.week,peak.rank,weeks.on.board
<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>
2021-11-06,1,Easy On Me,Adele,1,1,3
2021-11-06,2,Stay,The Kid LAROI & Justin Bieber,2,1,16
2021-11-06,3,Industry Baby,Lil Nas X & Jack Harlow,3,1,14
2021-11-06,4,Fancy Like,Walker Hayes,4,3,19
2021-11-06,5,Bad Habits,Ed Sheeran,5,2,18
2021-11-06,6,Way 2 Sexy,Drake Featuring Future & Young Thug,6,1,8
2021-11-06,7,Shivers,Ed Sheeran,9,7,7
2021-11-06,8,Good 4 U,Olivia Rodrigo,7,1,24
2021-11-06,9,Need To Know,Doja Cat,11,9,20


**Example.** For each artist, what is the most weeks that some song they had spent on the Billboard?

In [47]:
df |>
  group_by(artist) |>
  summarize(max(weeks.on.board)) |>
  head(10)

artist,max(weeks.on.board)
<chr>,<int>
"""Groove"" Holmes",11
"""Little"" Jimmy Dickens",10
"""Pookie"" Hudson",1
"""Weird Al"" Yankovic",20
'N Sync,26
'N Sync & Gloria Estefan,20
'N Sync Featuring Nelly,20
'Til Tuesday,21
(+44),1


In [49]:
# Sanity check: see if Ed Sheeran actually have that max number of weeks
 df |>
   filter(artist == "Ed Sheeran") |>
   group_by(artist) |>
   summarize(max(weeks.on.board), .groups = "drop") |>
   head(10)

 df |>
   filter(artist == "Ed Sheeran") |>
   arrange(desc(weeks.on.board)) |>
   head(3)

artist,max(weeks.on.board)
<chr>,<int>
Ed Sheeran,59


date,rank,song,artist,last.week,peak.rank,weeks.on.board
<chr>,<int>,<chr>,<chr>,<int>,<int>,<int>
2018-03-03,24,Shape Of You,Ed Sheeran,24,1,59
2018-02-24,24,Shape Of You,Ed Sheeran,23,1,58
2015-11-28,45,Thinking Out Loud,Ed Sheeran,41,2,58


## Big pictures


| Column | Row | Group (Do last) |
|--------|-----|-----------------|
| select | filter    | group_by  |
| mutate | drop_na   | summarize |
|        | distinct  | count     |
|        | slice     |           |
|        | arrange   |           |

# What else (optional topics, not required for class)

Learn to do joins in `dplyr`.

Learn more R packages:
- `tidyverse` loads `dplyr`, `stringr` (string manipulation), `readr`, `tibble`, `ggplot2` (graphs), etc.
- `lubridate` (date manipulation)
- `data.table` (`dplyr` alternative)
- `janitor` (use before `dplyr` to clean data)